# Stage 7 -- 细胞组成差异分析 (Differential Abundance)

本 notebook 做跨样本/跨条件的细胞组成差异分析，包含三个模块：

1. **细胞组成矩阵构建** -- 从 sample x cluster 计数矩阵计算各样本的细胞类型比例
2. **Mann-Whitney U 检验 + Cliff's Delta 效应量** -- 各 cluster 在不同条件下的比例差异
   非参数检验，不假设正态分布，适合小样本临床数据
3. **scCODA 贝叶斯组成差异分析** -- 基于 HMC 采样的贝叶斯模型，同时推断所有细胞类型
   的组成变化并输出 credible effect（后验包含概率 > 95% 的细胞类型）

**为什么做组成分析？** 单细胞数据中不同样本间细胞类型的相对丰度变化往往是揭示
疾病机制的第一把钥匙。例如胃"炎-癌"转化过程中，哪些细胞群在萎缩/肠化/异型增生
阶段出现显著扩增或缩减？scCODA 的贝叶斯框架能同时建模所有细胞类型的协变结构，
避免逐个检验的多重比较问题。

**重要提醒**：本 notebook 需要上游 stage6 的输出 h5ad，且需要 obs 中包含
条件（condition/group）列来做组间比较。当前 Nancang 夹具无此列，
Mann-Whitney 和 scCODA 模块会在检测到条件列缺失时**优雅跳过**并给出提示，
同时仍产出描述性组成统计（比例表 + 堆叠条形图）。

分析方法参考：scCODA (Büttner et al., 2021, *Nature Communications*)；
Mann-Whitney + Cliff's Delta + 效应量可视化重写自 student-code
`11_all_celltype_proportion_analyse.ipynb`，按本项目 notebook 规范完全重写
（ADR-0008：吸收算法思想，不复制代码）。

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：Stage 6（注释），读 `stage6_annotated_v*.h5ad`
- **下游**：下游分析（组成变化解读 / 跨条件对比），产出 `stage7_abundance_v*.h5ad` + 比例表 + scCODA 结果

### 为什么要迭代回跑？
细胞组成差异分析 (Abundance) 的结果是下游分析和 PI 生物学判断的基础。如果在后续分析中发现：
- DEG 阈值过高导致遗漏关键基因、过低导致假阳性
- 通路富集缺少预期应出现的生物学通路
- 调控网络缺少已知的主控转录因子
- CNV 信号不符合病理学预期
可能需要调整本 stage 的参数重新计算。

PI 的原始要求（来自项目构思）：
> "注释这一步，也可能在注释的过程中发现前面高可变基因的选择、embedding 的构建，
> 还有分群的参数等等需要调整，应可以随时调回去重跑一些流程，需要建立这种循环不断迭代的机制。"

### 如何回跑（三步操作）
1. **改 `UPSTREAM_PATH`**——指向要复用的上游文件版本
2. **改 `OUTPUT_PATH`**——bump 版本号 `_v1` → `_v2`（旧版不覆盖）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改以下参数后重跑本 notebook）：
- `CONDITION_COL` — 设置样本条件/分组列（必须先在 obs 中填充）
- `SCOODA_FORMULA` / `SCOODA_REF_CELL_TYPE` — scCODA 模型参数
- `SCOODA_QUICK_TEST` — True=快速验证（500采样），False=正式跑（20000）
- `MIN_CELLS_PER_CLUSTER` / `MIN_SAMPLES_PER_GROUP` — 过滤阈值

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。
  旧版 `.h5ad` 文件**不覆盖不删除**，保留在 `results/` 目录供追溯对比。
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）。
- **`promoted`**：PI 审查后认为结果可接受、可传给下游使用的正式版本。
  PI 在 Jupyter 中打开 `.h5ad` 后手动改 `adata.uns["status"] = "promoted"` 再保存。
- **下游取数**：下游 notebook 的 `UPSTREAM_PATH` 指向你决定采用的版本即可。

### 追溯链（自动写入 h5ad 的 `adata.uns`）
本 notebook 在写出前自动记录以下字段，供后续审计查询：
- `stage` = `"stage7_abundance"`（本 stage 标识）
- `status` = `"experimental"`（PI 审查后改为 `"promoted"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_PATH` 一致的版本号（`"v1"` / `"v2"` / ...）

如需查询「细胞组成差异分析 (Abundance) 有哪些版本？哪些依赖 stage6_v1？」，
可直接在 Python 中 glob `results/` 目录检查每个 `.h5ad` 的 `adata.uns`。

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH           -- stage6 注释结果 h5ad（需含 CLUSTER_KEY 或 CELL_TYPE_COL）
# OUTPUT_PATH         — 本 stage 产出 checkpoint 路径。
#                        版本号 _v1 与 adata.uns['version'] 保持一致。
#                        如需回跑：bump 版本号 _v1->v2，旧版不覆盖。
# CLUSTER_KEY             -- 用于细胞分组的 obs 列（cell_type_final_v1 全 NaN 时 fallback 到 leiden）
# CELL_TYPE_COL           -- 优先使用的细胞类型注释列（stage6 产出）
#                            重要提醒：当前 Nancang 夹具 cell_type_final_v1 全 NaN，
#                            因此自动 fallback 到 CLUSTER_KEY=leiden_res_0.6。
#                            PI 完成细胞类型注释后修改此参数。
# CONDITION_COL           -- 样本条件/分组列（如 disease_stage、group、condition）
#                            **重要提醒**：当前 Nancang 夹具无此列，设为 None。
#                            当 CONDITION_COL 缺失时，Mann-Whitney 和 scCODA 模块
#                            将优雅跳过（不崩 notebook），仅产出描述性组成统计。
#                            PI 获得含分组信息的真实数据后，请在 obs 中添加条件列
#                            并修改此参数（如 "disease_stage"、"group" 等）。
# SAMPLE_COL              -- 样本标识列
# SCOODA_FORMULA          -- scCODA 模型公式（条件协变量）
# SCOODA_REF_CELL_TYPE    -- scCODA reference 细胞类型（通常选最大/最稳定的类型）
# SCOODA_NUM_RESULTS      -- HMC 采样链长度（真实分析推荐 >= 20000）
# SCOODA_NUM_BURNIN       -- HMC 预热步数（真实分析推荐 >= 5000）
# SCOODA_QUICK_TEST       -- 快速验证模式：True 时用极小采样数快速跑通流程
#                            改为 False 使用 SCOODA_NUM_RESULTS/SCOODA_NUM_BURNIN 真实值
# MIN_CELLS_PER_CLUSTER   -- 每个 cluster 至少需在多少样本中出现才做检验
# MIN_SAMPLES_PER_GROUP   -- 每组至少需要多少个样本才能做组间比较

UPSTREAM_PATH = "results/nancang_stage6_annotated_v1.h5ad"
OUTPUT_PATH   = "results/stage7_abundance.h5ad"

CLUSTER_KEY   = "leiden_res_0.6"
CELL_TYPE_COL = "cell_type_final_v1"

CONDITION_COL = None  # 当前 Nancang 夹具无分组列；PI 在获得真实分组数据后修改
SAMPLE_COL    = "sample_id"

SCOODA_FORMULA       = "condition"        # 对应 obs 中的条件列名，与 CONDITION_COL 配合
SCOODA_REF_CELL_TYPE = None               # None=自动选零值最少的细胞类型作为 reference
SCOODA_NUM_RESULTS   = 20000              # 发表级别推荐 >= 20000
SCOODA_NUM_BURNIN    = 5000               # 发表级别推荐 >= 5000
SCOODA_QUICK_TEST    = True               # 快速验证：True 用 500/200；False 用真实值

MIN_CELLS_PER_CLUSTER = 3    # 至少在多少个样本中该 cluster 细胞数 > 0
MIN_SAMPLES_PER_GROUP = 3    # 每组至少 3 个样本才能做非参检验


In [ ]:
# 确保框架 src/ 在 sys.path 并切换到项目根目录。
# 多级回退策略：nbconvert/conda run 的 CWD 不稳定，
# 先试 CWD，再试从 notebooks/stage7/ 回退两级，最后用 notebook 自身路径推算。
import sys, os, gc as _gc
_root = os.getcwd()
_root_candidates = [
    _root,
    os.path.abspath(os.path.join(_root, "..")),
    os.path.abspath(os.path.join(_root, "..", "..")),
]

for _cand in _root_candidates:
    if os.path.isdir(os.path.join(_cand, "src", "scrna_integration")):
        _root = _cand
        break
else:
    _root = os.environ.get("PROJECT_ROOT", _root)

if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")
print(f"src 存在: {os.path.isdir(os.path.join(_root, 'src', 'scrna_integration'))}")


In [ ]:
# 导入依赖。
# sccoda 仅在有条件列时才实际使用，但仍在此导入以便静态检查。
import scanpy as sc
import scipy.sparse as sp
from scipy import stats
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
np.random.seed(42)

print(f"scanpy {sc.__version__}  |  numpy {np.__version__}  |  pandas {pd.__version__}")


In [ ]:
# 加载上游 stage6 输出。
# 契约：需包含 CLUSTER_KEY 或 CELL_TYPE_COL、SAMPLE_COL、
# 以及（若做组间比较）CONDITION_COL。
print(f"加载上游: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"obs 列: {list(adata.obs.columns)}")

# 确定实际使用的分组列（cell type / cluster column）
if CELL_TYPE_COL in adata.obs.columns and adata.obs[CELL_TYPE_COL].notna().sum() > 0:
    _group_col = CELL_TYPE_COL
    print(f"使用细胞类型列: {CELL_TYPE_COL}")
    _n_valid = adata.obs[CELL_TYPE_COL].notna().sum()
    print(f"  有效注释: {_n_valid}/{adata.n_obs} 细胞")
elif CLUSTER_KEY in adata.obs.columns:
    _group_col = CLUSTER_KEY
    print(f"CELL_TYPE_COL 无有效值，fallback 到 CLUSTER_KEY: {CLUSTER_KEY}")
    _n_clusters = adata.obs[CLUSTER_KEY].nunique()
    print(f"  簇数: {_n_clusters}")
else:
    raise KeyError(
        f"CELL_TYPE_COL '{CELL_TYPE_COL}' 和 "
        f"CLUSTER_KEY '{CLUSTER_KEY}' 都不在 obs 列中"
    )

# 检查 CONDITION_COL 可用性
_has_condition = (
    CONDITION_COL is not None
    and CONDITION_COL in adata.obs.columns
    and adata.obs[CONDITION_COL].notna().sum() > 0
)
print(f"CONDITION_COL '{CONDITION_COL}' 可用: {_has_condition}")
if _has_condition:
    _cond_vals = sorted(adata.obs[CONDITION_COL].dropna().unique())
    print(f"  条件值: {_cond_vals}")
    if len(_cond_vals) < 2:
        print(f"  WARNING: 条件列只有 {len(_cond_vals)} 个唯一值，无法做组间比较")
        _has_condition = False
else:
    _cond_skip_msg = (
        "=" * 60 + "\n"
        "CONDITION_COL 不可用（缺失或全为 NaN），以下模块将优雅跳过：\n"
        "  - Mann-Whitney U 检验 + Cliff's Delta 效应量\n"
        "  - scCODA 贝叶斯组成差异分析\n\n"
        "仍会产出描述性组成统计（比例表 + 堆叠条形图）。\n\n"
        "要启用组间差异分析，请在 obs 中填充条件列（如 disease_stage 或 group），\n"
        "并修改本 notebook PARAMS 中的 CONDITION_COL 参数。\n"
        "=" * 60
    )
    print(_cond_skip_msg)

# 确保 SAMPLE_COL 存在
if SAMPLE_COL not in adata.obs.columns:
    raise KeyError(f"SAMPLE_COL '{SAMPLE_COL}' 不在 obs 列中")

# 确保分组列是 str 类型
if not hasattr(adata.obs[_group_col], "cat"):
    adata.obs[_group_col] = adata.obs[_group_col].astype(str)
_groups_unique = sorted(adata.obs[_group_col].astype(str).unique())
print(f"分组值数: {len(_groups_unique)}")


## 1. 细胞组成矩阵构建

**为什么需要这个矩阵？** scCODA 和 Mann-Whitney 的输入都是"样本 x 细胞类型"的计数或比例表。
从单细胞级别的 obs 出发，按 sample_id 和 cluster/cell_type 做 groupby 聚合，
得到每个样本中每种细胞类型的绝对计数和相对比例。

**为什么用 `observed=True`？** 当 cluster 列是 pandas `CategoricalDtype` 时，
`groupby` 默认会输出所有可能的类别组合（包括计数为 0 的组合），产生大量"幽灵行"。
`observed=True` 只统计实际存在的组合，大幅减少 DataFrame 行数并消除下游分析的虚假零值。

关键列说明：
- `count_df`：长格式，每行一个 (sample, cluster, count, proportion)
- `counts_wide`：宽格式，行=样本，列=cluster，值=整数计数 -- scCODA 的输入格式
- `proportion_df`：宽格式，行=样本，列=cluster，值=百分比 -- Mann-Whitney 的输入格式


In [ ]:
# 构建样本 x 细胞类型计数矩阵（长表 + 宽表）。
# 为什么分长表和宽表？长表是 seaborn/plotly 绘图的标准输入格式；
# 宽表是 scCODA（样本 x 细胞类型）和 Mann-Whitney（每组各列）的输入格式。

obs = adata.obs.copy()

# Step 1：长表 count_df -- 每行一个 (sample, cluster) 组合
# observed=True 避免 pandas CategoricalDtype 产生幽灵行
count_df = (
    obs.groupby([SAMPLE_COL, _group_col], observed=True)
       .size()
       .reset_index(name="cell_count")
)

# 计算每个样本的总细胞数和各 cluster 比例
count_df["total_cells"] = count_df.groupby(
    SAMPLE_COL, observed=True
)["cell_count"].transform("sum")
count_df["proportion"] = count_df["cell_count"] / count_df["total_cells"] * 100

# 验证：每个样本的比例之和 = 100%
_pct_sum = count_df.groupby(SAMPLE_COL, observed=True)["proportion"].sum().round(1)
assert (_pct_sum == 100).all(), f"比例之和不为 100%: {_pct_sum[_pct_sum != 100].to_dict()}"
print("验证通过：每个样本比例之和 = 100%")

print(f"count_df 行数: {len(count_df)}，样本数: {count_df[SAMPLE_COL].nunique()}")
print(count_df.head(12).to_string(index=False))

# Step 2：宽表 counts_wide -- scCODA 输入（整数计数）
counts_wide = (
    count_df.pivot_table(
        index=SAMPLE_COL,
        columns=_group_col,
        values="cell_count",
        fill_value=0,
        observed=True,
    )
    .astype(int)
)
counts_wide.columns.name = None

# Step 3：宽表 proportion_df -- Mann-Whitney 输入（百分比）
proportion_df = (
    count_df.pivot_table(
        index=SAMPLE_COL,
        columns=_group_col,
        values="proportion",
        fill_value=0,
        observed=True,
    )
)
proportion_df.columns.name = None

# 若存在条件列，merge 到宽表中供 Mann-Whitney 分组
if _has_condition:
    _sample_meta = (
        obs[[SAMPLE_COL, CONDITION_COL]]
        .drop_duplicates()
        .set_index(SAMPLE_COL)
    )
    # 验证每个 sample 只对应一个 condition
    _dup_check = _sample_meta.groupby(SAMPLE_COL).nunique()
    if (_dup_check > 1).any().any():
        print("WARNING: 部分 sample 对应多个 condition 值，取第一个")
        _sample_meta = _sample_meta[~_sample_meta.index.duplicated(keep="first")]
    proportion_df = proportion_df.merge(
        _sample_meta, left_index=True, right_index=True, how="left"
    )

print(f"counts_wide 维度: {counts_wide.shape}  (样本 x cluster)")
print(f"proportion_df 维度: {proportion_df.shape}")
print(f"cluster 列表: {list(counts_wide.columns)}")

# 保存计数矩阵
counts_csv = "results/tables/stage7_abundance_counts_wide.csv"
counts_wide.to_csv(counts_csv)
print(f"计数宽表已保存: {counts_csv}")

prop_csv = "results/tables/stage7_abundance_proportions.csv"
proportion_df.to_csv(prop_csv)
print(f"比例表已保存: {prop_csv}")


## 2. 组成可视化

**堆叠条形图**：每个样本的细胞组成一览。横轴=样本，纵轴=比例，每个颜色段=一个 cluster。
可用肉眼快速判断：(1) 是否存在样本间的整体组成漂移；
(2) 某些 cluster 是否仅在特定样本中出现或占比异常。

**为什么按样本而非按条件分组？** 在当前 Nancang 夹具中无条件列，
堆叠条形图以样本为粒度展示组成异质性。当 PI 添加条件列后，
可在 x 轴上按条件分面或添加分隔线。

In [ ]:
# 堆叠柱状图 -- 每个样本的细胞组成可视化。
# 颜色方案：使用 tab20 色板（足够 20 种细胞类型），
# 当 cluster 数 > 20 时 fallback 到连续色板。

_cell_types_list = sorted(counts_wide.columns.tolist())
_n_types = len(_cell_types_list)

# 颜色方案
if _n_types <= 10:
    _cmap = plt.cm.tab10
elif _n_types <= 20:
    _cmap = plt.cm.tab20
else:
    _cmap = plt.cm.viridis

_ct_colors = {
    ct: _cmap(i / max(_n_types - 1, 1)) for i, ct in enumerate(_cell_types_list)
}

# 按 sample_id 保持自然顺序
_sample_order = sorted(counts_wide.index.tolist())

fig, ax = plt.subplots(figsize=(max(8, _n_types * 0.3), 5))

# 透视为堆叠图格式
_pivot_plot = proportion_df[_cell_types_list].reindex(_sample_order)

bottom = np.zeros(len(_pivot_plot))
for ct in _cell_types_list:
    vals = _pivot_plot[ct].values
    ax.bar(
        range(len(_pivot_plot)),
        vals,
        bottom=bottom,
        color=_ct_colors[ct],
        label=f"Cluster {ct}",
        width=0.75,
        edgecolor="white",
        linewidth=0.5,
    )
    bottom += vals

# 图形美化
ax.set_xticks(range(len(_pivot_plot)))
ax.set_xticklabels(_pivot_plot.index, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Cell Type Proportion (%)", fontsize=12)
ax.set_xlabel("Sample", fontsize=12)
ax.set_title(
    f"Cell Type Composition per Sample (grouped by {_group_col})",
    fontsize=13, fontweight="bold", pad=20,
)
ax.set_ylim(0, 105)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# 图例放右侧，当 cluster 多时缩小字体
_legend_fontsize = 8 if _n_types > 12 else 10
ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    title=f"{_group_col}",
    title_fontsize=_legend_fontsize,
    fontsize=_legend_fontsize - 1,
    frameon=False,
)

plt.tight_layout()
_stack_path_png = "results/figures/stage7_abundance_stacked_bar.png"
_stack_path_pdf = "results/figures/stage7_abundance_stacked_bar.pdf"
fig.savefig(_stack_path_png, dpi=200, bbox_inches="tight")
fig.savefig(_stack_path_pdf, dpi=200, bbox_inches="tight")
plt.close("all")
print(f"堆叠柱状图已保存: {_stack_path_png}")
print(f"  样本数: {len(_sample_order)}  |  cluster 数: {_n_types}")

# 快速预览：每个 cluster 在各样本中的比例分布
print(f"\n各 cluster 比例统计（跨 {len(_sample_order)} 个样本）:")
print(proportion_df[_cell_types_list].describe().round(2).to_string())


## 3. 组间差异检验（Mann-Whitney U + Cliff's Delta）

**为什么用 Mann-Whitney U 而非 t 检验？** 单细胞数据的细胞类型比例通常不服从
正态分布（小样本 + 比例数据天然有界 [0, 100]）。Mann-Whitney U 是秩和检验，
不假设分布形态，适合小样本独立双组比较（n >= 3 per group）。

**为什么配合 Cliff's Delta 而非 Cohen's d？** Cliff's Delta 是非参数效应量，
不假设正态性和方差齐性，与 Mann-Whitney U 一致。其值域为 [-1, +1]，解读标准
（Romano 2006）：|delta| < 0.147 negligible，< 0.330 small，< 0.474 medium，>= 0.474 large。

**多重检验校正**：同时对多个 cluster 做检验，用 FDR-BH 方法校正 p 值。
图中显著性标注基于校正后的 p_adj，而非原始 p 值。
标注：* p_adj < 0.05，** p_adj < 0.01，*** p_adj < 0.001。

此模块重写自 student-code `11_all_celltype_proportion_analyse.ipynb` Step 4，
按本项目规范简化：去硬编码路径、去 Windows-only 逻辑、注释中文化、变量命名统一。

In [ ]:
# Mann-Whitney U 检验 + Cliff's Delta 效应量。
# 仅当 CONDITION_COL 可用且每组 >= MIN_SAMPLES_PER_GROUP 样本时执行。

_mw_ran = False  # 标记 Mann-Whitney 是否实际运行
_n_sig = 0  # 显式初始化，确保即使 MW 未运行也不会 NameError

if not _has_condition:
    print(
        "CONDITION_COL 不可用，跳过 Mann-Whitney U 检验。\n"
        "要启用此分析，请：\n"
        "  1. 在 obs 中填充条件列（如 disease_stage、group 等）\n"
        "  2. 修改 PARAMS 中的 CONDITION_COL\n"
        "  3. 重跑本 notebook"
    )
else:
    # 检查每组样本数
    if SAMPLE_COL in proportion_df.columns:
        _cond_samples = proportion_df.groupby(CONDITION_COL)[SAMPLE_COL].nunique()
    else:
        _cond_samples = proportion_df.groupby(CONDITION_COL).size()
    print(f"各组样本数: {_cond_samples.to_dict()}")

    _groups = list(_cond_samples.index)
    if len(_groups) < 2:
        print(f"条件列只有 {len(_groups)} 组，无法做组间比较，跳过 Mann-Whitney U")
    elif any(n < MIN_SAMPLES_PER_GROUP for n in _cond_samples):
        print(
            f"部分组样本数不足 (< {MIN_SAMPLES_PER_GROUP})，\n"
            f"跳过 Mann-Whitney U 以避免不可靠的小样本推断"
        )
    else:
        # Cliff's Delta 效应量函数
        def _cliffs_delta(a, b):
            '''计算 Cliff’s Delta 效应量（非参数，配合 Mann-Whitney U）。
            定义：随机抽一个 A 组和一个 B 组，A > B 的概率减 B > A 的概率。
            返回值范围 [-1, +1]，正值表示 A 组倾向更大。'''
            a, b = np.array(a), np.array(b)
            n_a, n_b = len(a), len(b)
            more = np.sum(a[:, None] > b[None, :])
            less = np.sum(a[:, None] < b[None, :])
            return (more - less) / (n_a * n_b)

        def _interpret_delta(d):
            abs_d = abs(d)
            if abs_d >= 0.474:
                return "large"
            elif abs_d >= 0.330:
                return "medium"
            elif abs_d >= 0.147:
                return "small"
            else:
                return "negligible"

        _cell_types = [
            c for c in proportion_df.columns
            if c not in [SAMPLE_COL, CONDITION_COL]
        ]

        # 确定两个比较组
        _g0, _g1 = _groups[0], _groups[1]
        print(f"比较组: {_g0} vs {_g1}")

        _stats_results = []
        for ct in _cell_types:
            # 过滤该 cluster 至少在有 MIN_CELLS_PER_CLUSTER 个样本中有 >0 比例
            _mask = proportion_df[ct] > 0
            _n_nonzero = _mask.sum()
            if _n_nonzero < MIN_CELLS_PER_CLUSTER:
                print(f"  {ct}: 仅 {_n_nonzero} 个样本有非零比例，跳过检验")
                continue

            _v0 = proportion_df.loc[
                proportion_df[CONDITION_COL] == _g0, ct
            ].dropna().values
            _v1 = proportion_df.loc[
                proportion_df[CONDITION_COL] == _g1, ct
            ].dropna().values

            if len(_v0) < 2 or len(_v1) < 2:
                print(f"  {ct}: 样本量不足，跳过检验")
                continue

            _stat, _pval = stats.mannwhitneyu(_v0, _v1, alternative="two-sided")
            _delta = _cliffs_delta(_v0, _v1)

            _stats_results.append({
                "cell_type": ct,
                f"n_{_g0}": len(_v0),
                f"n_{_g1}": len(_v1),
                f"mean_{_g0}": round(np.mean(_v0), 2),
                f"mean_{_g1}": round(np.mean(_v1), 2),
                "U_statistic": _stat,
                "p_value": _pval,
                "cliffs_delta": round(_delta, 4),
                "delta_interpretation": _interpret_delta(_delta),
            })

        if _stats_results:
            _stats_df = pd.DataFrame(_stats_results)

            # FDR-BH 多重检验校正
            # FDR-BH 多重检验校正 (scipy.stats.false_discovery_control, scipy>=1.11)
            _p_adj = stats.false_discovery_control(
                _stats_df["p_value"].values, method='bh'
            )
            _reject = _p_adj < 0.05
            _stats_df["p_adj"] = _p_adj
            _stats_df["significant"] = _reject

            # 按 p_adj 升序排列
            _stats_df = _stats_df.sort_values("p_adj").reset_index(drop=True)

            # 显著性符号（基于校正后 p 值）
            def _sig_label(p):
                if p < 0.001:
                    return "***"
                elif p < 0.01:
                    return "**"
                elif p < 0.05:
                    return "*"
                else:
                    return "ns"

            _stats_df["significance"] = _stats_df["p_adj"].apply(_sig_label)

            print("\nMann-Whitney U 检验结果（FDR-BH 校正，按 p_adj 升序）:")
            print(_stats_df[[
                "cell_type",
                f"mean_{_g0}", f"mean_{_g1}",
                "p_value", "p_adj", "significance",
                "cliffs_delta", "delta_interpretation",
            ]].to_string(index=False))

            # 保存统计结果
            _stats_csv = "results/tables/stage7_abundance_mannwhitney.csv"
            _stats_df.to_csv(_stats_csv, index=False)
            print(f"统计结果已保存: {_stats_csv}")

            # 将显著差异的 cluster 数写入
            _n_sig = _stats_df["significant"].sum()
            print(f"显著差异 cluster 数 (p_adj < 0.05): {_n_sig}/{len(_stats_df)}")
            _mw_ran = True
        else:
            print("无有效的 cluster 可做检验")


### 箱线图：各 cluster 比例组间比较

重写自 student-code `11_all_celltype_proportion_analyse.ipynb` 的箱线图矩阵，
按本项目规范简化：去硬编码配色、自适应子图布局、用 p_adj 标注显著性。

In [ ]:
# 各 cluster 组间比例箱线图矩阵。
# 仅当 Mann-Whitney 结果存在且 _mw_ran=True 时绘制。

if _mw_ran:
    _ncols_box = min(4, len(_cell_types))
    _nrows_box = int(np.ceil(len(_cell_types) / _ncols_box))
    fig_box, axes_box = plt.subplots(
        _nrows_box, _ncols_box,
        figsize=(_ncols_box * 3.2, _nrows_box * 3.2),
    )
    axes_box = np.atleast_1d(axes_box).flatten()

    _palette = {_g0: "#E53935", _g1: "#1E88E5"}
    _alpha = 0.05
    _y_min_global = float("inf")
    _y_max_global = float("-inf")

    # 转换为长表供 seaborn 绘图
    _long_df = proportion_df.melt(
        id_vars=[CONDITION_COL],
        value_vars=_cell_types,
        var_name="cell_type",
        value_name="proportion",
    )

    for idx, ct in enumerate(_cell_types):
        ax = axes_box[idx]
        _sub = _long_df[_long_df["cell_type"] == ct]

        # 箱线图
        sns.boxplot(
            data=_sub, x=CONDITION_COL, y="proportion",
            palette=_palette, width=0.5, linewidth=1.2,
            flierprops=dict(marker="o", markersize=4),
            ax=ax,
        )
        # 散点叠加（展示真实样本分布）
        sns.stripplot(
            data=_sub, x=CONDITION_COL, y="proportion",
            color="black", size=5, alpha=0.8, jitter=True, ax=ax,
        )

        # 显著性标注（基于 p_adj）
        _row = _stats_df[_stats_df["cell_type"] == ct]
        if len(_row) > 0:
            _p = _row["p_adj"].values[0]
            _d = _row["cliffs_delta"].values[0]
            _di = _row["delta_interpretation"].values[0]
            _y_data = _sub["proportion"]
            _y_max = _y_data.max()
            _y_min = _y_data.min()
            _y_min_global = min(_y_min_global, _y_min)
            _y_max_global = max(_y_max_global, _y_max)

            _y_gap = max(_y_max * 0.15, 1.5)

            if _p < 0.001:
                _sig_text = "***"
            elif _p < 0.01:
                _sig_text = "**"
            elif _p < 0.05:
                _sig_text = "*"
            else:
                _sig_text = f"p={_p:.2f}"

            _y_line = _y_max + _y_gap * 0.3
            _y_text = _y_max + _y_gap * 0.5
            ax.plot(
                [0, 0, 1, 1],
                [_y_line, _y_line + _y_gap * 0.2,
                 _y_line + _y_gap * 0.2, _y_line],
                color="black", linewidth=1,
            )
            ax.text(
                0.5, _y_text, _sig_text,
                ha="center", va="bottom", fontsize=11,
                color="red" if _p < _alpha else "black",
            )
            ax.text(
                0.5, -0.18,
                f"delta={_d:.2f} ({_di})",
                ha="center", va="top", fontsize=7.5,
                color="#555555", transform=ax.transAxes,
            )

        ax.set_title(f"Cluster {ct}", fontsize=11, fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("Proportion (%)" if idx % _ncols_box == 0 else "")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # 隐藏多余子图
    for i in range(len(_cell_types), len(axes_box)):
        axes_box[i].set_visible(False)

    # 统一 y 轴范围
    _y_range = max(_y_max_global - _y_min_global, 1.0)
    _y_pad = _y_range * 0.25
    for i in range(len(_cell_types)):
        axes_box[i].set_ylim(
            bottom=max(0, _y_min_global - _y_pad),
            top=_y_max_global + _y_pad * 2,
        )

    plt.suptitle(
        f"Cell Type Proportion: {_g0} vs {_g1}\n"
        "(* p_adj < 0.05, ** p_adj < 0.01, *** p_adj < 0.001; "
        "FDR-BH correction; delta = Cliff's Delta)",
        fontsize=11, fontweight="bold", y=1.02,
    )
    plt.tight_layout()

    _box_png = "results/figures/stage7_abundance_boxplot.png"
    _box_pdf = "results/figures/stage7_abundance_boxplot.pdf"
    fig_box.savefig(_box_png, dpi=200, bbox_inches="tight")
    fig_box.savefig(_box_pdf, dpi=200, bbox_inches="tight")
    plt.close("all")
    print(f"箱线图已保存: {_box_png}")
else:
    print("无 Mann-Whitney 结果，跳过箱线图")


### 效应量可视化：Log2 Fold Change 水平条形图

展示每个 cluster 在两组间的中位比例差异（log2 fold change），
同时标注 FDR-BH 校正后的显著性符号和 Cliff's Delta 效应量。
正值表示在比较组 A 中富集，负值表示在比较组 B 中富集。

In [ ]:
# Log2 Fold Change 水平条形图。
# 中位比例差异 + Cliff's Delta + 显著性标注。
# 重写自 student-code Step 4 末尾的 log2fc 条形图。

if _mw_ran:
    # 计算各组中位比例
    _median_props = proportion_df.groupby(CONDITION_COL)[_cell_types].median()
    _epsilon = 0.01
    _log2fc = np.log2(
        (_median_props.loc[_g0] + _epsilon)
        / (_median_props.loc[_g1] + _epsilon)
    )

    _fc_df = pd.DataFrame({
        "cell_type": _log2fc.index,
        "log2fc": _log2fc.values,
        "direction": [
            f"Enriched in {_g0}" if x > 0 else f"Depleted in {_g0}"
            for x in _log2fc.values
        ],
    }).merge(
        _stats_df[["cell_type", "p_value", "p_adj", "cliffs_delta",
                    "delta_interpretation", "significance"]],
        on="cell_type", how="left",
    ).sort_values("log2fc", ascending=True).reset_index(drop=True)

    print("细胞比例变化汇总（按 Log2FC 升序，图中从下到上）:")
    print(_fc_df.to_string(index=False))

    # 绘制水平条形图
    fig_fc, ax_fc = plt.subplots(figsize=(9, max(4, len(_fc_df) * 0.4)))
    _bar_colors = [
        "#E53935" if x > 0 else "#1E88E5" for x in _fc_df["log2fc"]
    ]
    ax_fc.barh(
        _fc_df["cell_type"], _fc_df["log2fc"],
        color=_bar_colors, alpha=0.85,
        edgecolor="white", linewidth=0.8,
    )

    # 标注显著性 + Cliff's Delta
    _x_range = _fc_df["log2fc"].abs().max()
    if _x_range < 0.1:
        _x_range = 0.1  # 避免零范围
    _offset = _x_range * 0.05
    for i, row in _fc_df.iterrows():
        _lfc = row["log2fc"]
        _sig = row["significance"]
        _d = row["cliffs_delta"]
        _xpos = _lfc + _offset if _lfc >= 0 else _lfc - _offset
        _ha = "left" if _lfc >= 0 else "right"
        _color = "red" if _sig != "ns" else "#777777"
        _weight = "bold" if _sig != "ns" else "normal"
        _label = f"{_sig}  delta={_d:+.2f}"
        ax_fc.text(_xpos, i, _label, va="center", ha=_ha,
                   fontsize=9, color=_color, fontweight=_weight)

    ax_fc.axvline(x=0, color="black", linewidth=1.2, linestyle="-")
    ax_fc.set_xlabel("log2 Fold Change", fontsize=12)
    ax_fc.set_title(
        f"Cell Type Compositional Changes: {_g0} vs {_g1}\n"
        "(Significance: FDR-BH corrected p_adj; delta = Cliff's Delta)",
        fontsize=11, fontweight="bold",
    )
    ax_fc.spines["top"].set_visible(False)
    ax_fc.spines["right"].set_visible(False)
    ax_fc.grid(axis="x", linestyle="--", alpha=0.3)

    _x_min = _fc_df["log2fc"].min()
    _x_max = _fc_df["log2fc"].max()
    ax_fc.set_xlim(_x_min - _x_range * 0.4, _x_max + _x_range * 0.5)

    # 图例
    from matplotlib.patches import Patch as _Patch
    _legend_el = [
        _Patch(color="#E53935", alpha=0.85, label=f"Enriched in {_g0}"),
        _Patch(color="#1E88E5", alpha=0.85, label=f"Depleted in {_g0}"),
    ]
    ax_fc.legend(handles=_legend_el, loc="lower right",
                 frameon=True, framealpha=0.9, fontsize=9)

    plt.tight_layout()
    _fc_png = "results/figures/stage7_abundance_log2fc.png"
    _fc_pdf = "results/figures/stage7_abundance_log2fc.pdf"
    fig_fc.savefig(_fc_png, dpi=200, bbox_inches="tight")
    fig_fc.savefig(_fc_pdf, dpi=200, bbox_inches="tight")
    plt.close("all")
    print(f"Log2FC 条形图已保存: {_fc_png}")

    _fc_csv = "results/tables/stage7_abundance_log2fc_summary.csv"
    _fc_df.to_csv(_fc_csv, index=False)
    print(f"汇总表已保存: {_fc_csv}")
else:
    print("无统计结果，跳过效应量可视化")


## 4. scCODA 贝叶斯组成差异分析

**为什么用 scCODA？** 传统方法对每个细胞类型独立做 t/Mann-Whitney 检验，
然后用 Bonferroni/FDR 校正多重比较。问题在于：
(1) 细胞类型比例之间存在天然的负相关（如果一个类型增加了，其他类型必然
   相对减少），独立检验忽略了这种协变结构；
(2) 小样本下独立检验的统计功效很低。

scCODA (Büttner et al., 2021, *Nature Communications*) 用贝叶斯层次模型
同时对所有细胞类型建模：以 Dirichlet-Multinomial 为观测模型，
用 HMC (Hamiltonian Monte Carlo) 采样推断每个细胞类型在条件间的
对数折叠变化（log-fold change vs reference cell type）及其后验包含概率。
最终输出 **credible effects** -- 后验包含概率高于阈值的细胞类型，
即"有可信证据表明其组成在条件间发生了显著变化"。

**为什么需要 reference cell type？** Dirichlet-Multinomial 的组成约束
导致各细胞类型的变化不是独立的（比例之和必须为 1）。scCODA 选择一个
"不变化"的 reference 细胞类型来锚定模型，其他细胞类型的效应解释为
"相对 reference 的变化"。通常选数量最多/最稳定的细胞类型。

**HMC 采样注意事项**：采样链长度（num_results）和预热步数（num_burnin）
决定了后验推断的质量。发表级别推荐 num_results >= 20000。
快速验证时 SCOODA_QUICK_TEST=True 使用 500/200，确认跑通后改为 False。

**scCODA 守卫策略**：
1. 条件列缺失 → 跳过，给出清晰提示
2. 样本数/组数不足 → 跳过，提示增加样本
3. sccoda 包未安装 → 跳过，给出安装命令
4. HMC 采样发散 → 捕获异常，给出调参建议

此模块重写自 student-code `11_all_celltype_proportion_analyse.ipynb` 的
scCODA 段（Step 1-5），按本项目规范完全重写：去 `!pip install` cell、
去 `pip install -U` 裸调用、去硬编码 Windows 路径、去 arviz 降级 hack、
注释中文化。

In [ ]:
# scCODA 贝叶斯组成差异分析。
# 守卫检查：条件列 > sccoda 包 > 样本量 > reference 选择 > HMC 采样。

_sccoda_available = False
_sccoda_ran = False

# Guard 1: 条件列可用
if not _has_condition:
    print(
        "CONDITION_COL 不可用，跳过 scCODA 分析。\n"
        "scCODA 需要条件列来做贝叶斯组成差异推断。\n"
        "要启用，请：\n"
        "  1. 在 obs 中添加条件列（如 disease_stage、group 等）\n"
        "  2. 修改 PARAMS 中的 CONDITION_COL\n"
        "  3. 确保条件列有 >= 2 个组，每组 >= 3 个样本\n"
        "  4. 重跑本 notebook"
    )
else:
    # Guard 2: sccoda 包可用
    try:
        import sccoda
        from sccoda.util import cell_composition_data as scdat
        from sccoda.util import comp_ana as scmod
        from sccoda.util import data_visualization as scviz
        _sccoda_available = True
        print(f"sccoda {sccoda.__version__} 可用")
    except ImportError as _e:
        print(
            f"sccoda 未安装 ({_e})，跳过 scCODA 分析。\n"
            "要安装: conda run -n scrna-integration pip install sccoda"
        )

if _sccoda_available:
    # Guard 3: 样本量检查
    _cond_counts = proportion_df.groupby(CONDITION_COL).size()
    _n_groups = len(_cond_counts)
    if _n_groups < 2:
        print(
            f"条件列只有 {_n_groups} 组，"
            f"scCODA 需要 >= 2 组，跳过"
        )
        _sccoda_available = False
    elif any(n < 3 for n in _cond_counts):
        print(
            f"部分组样本数不足 (< 3): {_cond_counts.to_dict()}，\n"
            "scCODA 贝叶斯推断在极端小样本下发散风险高，跳过。\n"
            "建议: 每组 >= 5 个样本再启用 scCODA。"
        )
        _sccoda_available = False

if _sccoda_available:
    # 构建 scCODA 输入 DataFrame
    # 确定 reference cell type
    if SCOODA_REF_CELL_TYPE is None:
        # 自动选：零值样本数最少的 cell type（最稳定/最普遍）
        _zero_counts = (counts_wide == 0).sum(axis=0)
        _ref_ct = _zero_counts.idxmin()
        print(
            f"自动选择 reference cell type: {_ref_ct} "
            f"(零值样本数: {_zero_counts[_ref_ct]}/{len(counts_wide)})"
        )
    else:
        _ref_ct = SCOODA_REF_CELL_TYPE
        if _ref_ct not in counts_wide.columns:
            _zero_counts = (counts_wide == 0).sum(axis=0)
            _ref_ct = _zero_counts.idxmin()
            print(
                f"指定的 reference '{SCOODA_REF_CELL_TYPE}' "
                f"不在细胞类型列表中，改用: {_ref_ct}"
            )
        print(f"reference cell type: {_ref_ct}")

    # 合并 counts + meta
    _sccoda_df = counts_wide.copy()
    _sample_meta = (
        obs[[SAMPLE_COL, CONDITION_COL]]
        .drop_duplicates()
        .set_index(SAMPLE_COL)
    )
    _sccoda_df.insert(0, CONDITION_COL, _sample_meta[CONDITION_COL])

    print(f"scCODA 输入 DataFrame: {_sccoda_df.shape}")
    print(_sccoda_df.head().to_string())

    # 构建 sccoda data
    try:
        sccoda_data = scdat.from_pandas(
            _sccoda_df.reset_index(),
            covariate_columns=[SAMPLE_COL, CONDITION_COL],
        )
        print(
            f"sccoda_data: {sccoda_data.n_obs} 样本, "
            f"{sccoda_data.n_vars} 细胞类型"
        )
    except Exception as _e:
        print(f"sccoda from_pandas 失败: {_e}，跳过")
        _sccoda_available = False

if _sccoda_available:
    # scCODA 模型构建 + HMC 采样
    try:
        _formula = CONDITION_COL
        model = scmod.CompositionalAnalysis(
            sccoda_data,
            formula=_formula,
            reference_cell_type=_ref_ct,
        )
        print(
            f"scCODA 模型已构建: formula='{_formula}', "
            f"reference='{_ref_ct}'"
        )

        # 快速验证 vs 真实跑
        if SCOODA_QUICK_TEST:
            _hmc_results = 500
            _hmc_burnin = 200
            print("快速验证模式: num_results=500, num_burnin=200")
        else:
            _hmc_results = SCOODA_NUM_RESULTS
            _hmc_burnin = SCOODA_NUM_BURNIN
            print(
                f"HMC 采样: num_results={_hmc_results}, "
                f"num_burnin={_hmc_burnin}"
            )

        results = model.sample_hmc(
            num_results=_hmc_results,
            num_burnin=_hmc_burnin,
        )
        print("scCODA HMC 采样完成")
        _sccoda_ran = True
    except Exception as _e:
        print(
            f"scCODA HMC 采样失败: {_e}\n"
            "常见原因: (1) 样本量不足导致后验发散; "
            "(2) reference cell type 选择不当; "
            "(3) 条件组间差异过小。\n"
            "建议: 增加样本量，或检查条件组的生物学合理性。"
        )


In [ ]:
# 提取 scCODA 结果并可视化。
# credible_effects(): 返回后验包含概率高于阈值的细胞类型列表。

# 独立提取组名（不依赖 Mann-Whitney cell 的 _g0/_g1），确保本 cell 可独立重跑。
if _has_condition and CONDITION_COL is not None and CONDITION_COL in proportion_df.columns:
    _groups_sc = sorted(proportion_df[CONDITION_COL].dropna().unique())
    if len(_groups_sc) >= 2:
        _g0, _g1 = _groups_sc[:2]
    else:
        _g0, _g1 = None, None
        warnings.warn('scCODA results: < 2 condition groups, labels unavailable')
else:
    _g0, _g1 = None, None

if _sccoda_ran:
    # 摘要
    _summary_str = str(results.summary())
    _summary_path = "results/tables/stage7_abundance_sccoda_summary.txt"
    with open(_summary_path, "w", encoding="utf-8") as f:
        f.write(_summary_str)
    print(f"scCODA summary 已保存: {_summary_path}")
    # 打印前面部分供预览
    _preview_lines = _summary_str[:1500].split("\n")
    print("\n".join(_preview_lines[:25]))

    # 可信效应
    try:
        _ce_raw = results.credible_effects()

        # 统一转为 DataFrame
        if isinstance(_ce_raw, pd.DataFrame):
            _ce_df = _ce_raw.copy()
        elif isinstance(_ce_raw, pd.Series):
            _ce_df = _ce_raw.reset_index()
            _ce_df.columns = ["cell_type", "is_credible"]
        elif isinstance(_ce_raw, (list, np.ndarray)):
            _ce_df = pd.DataFrame({
                "cell_type": list(_ce_raw),
                "is_credible": True,
            })
        else:
            _ce_df = pd.DataFrame({
                "cell_type": [str(_ce_raw)],
                "is_credible": True,
            })

        print(f"\nCredible effects ({len(_ce_df)} 细胞类型):")
        print(_ce_df.to_string(index=False))

        _ce_csv = (
            "results/tables/"
            "stage7_abundance_sccoda_credible_effects.csv"
        )
        _ce_df.to_csv(_ce_csv, index=False)
        print(f"Credible effects 已保存: {_ce_csv}")

        # 内置可视化：效应量图
        fig_eff, ax_eff = plt.subplots(figsize=(8, 5))
        try:
            scviz.effects_barplot(results)
            plt.title(
                "scCODA: Credible Compositional Changes\n"
                f"({CONDITION_COL}: {_g0} vs {_g1})",
                fontsize=12, fontweight="bold",
            )
            plt.tight_layout()
            _eff_png = (
                "results/figures/"
                "stage7_abundance_sccoda_effects.png"
            )
            _eff_pdf = (
                "results/figures/"
                "stage7_abundance_sccoda_effects.pdf"
            )
            fig_eff.savefig(_eff_png, dpi=200, bbox_inches="tight")
            fig_eff.savefig(_eff_pdf, dpi=200, bbox_inches="tight")
            plt.close("all")
            print(f"scCODA 效应量图已保存: {_eff_png}")
        except Exception as _e:
            print(f"scviz.effects_barplot 失败 ({_e})，跳过")
            plt.close("all")

    except Exception as _e:
        print(f"提取 scCODA 结果失败: {_e}")
else:
    print("scCODA 未运行，跳过结果提取")


In [ ]:
# 内存自检 -- 确保 adata.X 稀疏性/精度未被破坏。
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量被破坏: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 是 sparse CSR float32")

# 写入 adata.uns 供下游 notebook 消费
_abundance_uns = {
    "group_col": _group_col,
    "condition_col": CONDITION_COL if _has_condition else None,
    "sample_col": SAMPLE_COL,
    "counts_wide_csv": counts_csv,
    "proportions_csv": prop_csv,
    "sccoda_ran": _sccoda_ran,
    "mannwhitney_ran": _mw_ran,
    "timestamp": __import__("datetime").datetime.now().isoformat(),
}

# 将 composition stats 写为 adata.uns 供后续使用
if _mw_ran:
    _abundance_uns["n_significant_clusters"] = int(_n_sig)

# 统一追踪字段——stage + version（与上游字段合并，保持 pipeline 命名一致）
adata.uns["stage"] = "stage7_abundance"     # 本 stage 标识
adata.uns["version"] = "v1"                  # 与 OUTPUT_PATH 版本号一致
adata.uns["upstream"] = [UPSTREAM_PATH]
adata.uns["status"] = "experimental"  # PI 审查后改为 "promoted"


adata.uns["stage7_abundance_v1"] = _abundance_uns
print("运行元数据已写入 adata.uns['stage7_abundance_v1']")


In [ ]:
# 写出 checkpoint。
# 注意：本 notebook 不修改 adata.X/adata.obs 本身（不做细胞级标注），
# 产出是 tables + figures + adata.uns 中的元数据。
# 因此 checkpoint h5ad 与上游基本相同，仅 adata.uns 新增 stage7_abundance_v1。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写出 {OUTPUT_PATH}")
assert os.path.exists(OUTPUT_PATH), f"输出不存在: {OUTPUT_PATH}"
print(f"已验证: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")


In [ ]:
# 释放内存。
del adata
_gc.collect()
print("内存已释放")
